In [ ]:
# Scaled Dot-Product Attention 标度点积注意力

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

def scaled_dot_product_attention(query, key, value, mask=None):
    """
    计算 Scaled Dot-Product Attention
    query: (batch_size, num_heads, seq_len_q, depth)
    key: (batch_size, num_heads, seq_len_k, depth)
    value: (batch_size, num_heads, seq_len_v, depth_v)
    mask: (batch_size, 1, seq_len_q, seq_len_k) or None
    """
    # 计算点积（矩阵乘法）
    matmul_qk = torch.matmul(query, key.transpose(-2, -1))  # (batch_size, num_heads, seq_len_q, seq_len_k)

    # 缩放
    dk = key.size()[-1]
    scores = matmul_qk / math.sqrt(dk)

    # 添加掩码（如果有）
    if mask is not None:
        scores += (mask * -1e9)  # 将掩码位置的值设为负无穷大

    # 计算注意力权重
    attention_weights = F.softmax(scores, dim=-1)  # (batch_size, num_heads, seq_len_q, seq_len_k)

    # 计算输出
    output = torch.matmul(attention_weights, value)  # (batch_size, num_heads, seq_len_q, depth_v)

    return output, attention_weights

In [ ]:
# MHA (Multi-Head Attention) 多头注意力机制

class MHA(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        self.head_dim = d_model // num_heads

        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.w_out = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        batch_size = x.size(0)

        # 线性变换
        query = self.wq(x)  # (batch_size, seq_len, d_model)
        key = self.wk(x)    # (batch_size, seq_len, d_model)
        value = self.wv(x)  # (batch_size, seq_len, d_model)

        # 分割为多头
        query = query.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)  # (batch_size, num_heads, seq_len, head_dim)
        key = key.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)      # (batch_size, num_heads, seq_len, head_dim)
        value = value.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)  # (batch_size, num_heads, seq_len, head_dim)

        # 计算注意力
        output, attention_weights = scaled_dot_product_attention(query, key, value, mask)

        # 合并多头
        output = output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)  # (batch_size, seq_len, d_model)

        return self.w_out(output)
        

In [ ]:
# MQA (Multi-Query Attention) 多查询注意力机制

class MQA(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        self.head_dim = d_model // num_heads

        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, self.head_dim)
        self.wv = nn.Linear(d_model, self.head_dim)
        self.w_out = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        B, S, _ = x.shape

        q = self.wq(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.wk(x).view(B, S, 1, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(B, S, 1, self.head_dim).transpose(1, 2)

        # repeat k and v for each head
        k = k.expand(-1, self.num_heads, -1, -1)
        v = v.expand(-1, self.num_heads, -1, -1)

        out = scaled_dot_product_attention(q, k, v, mask)
        out = out.transpose(1, 2).contiguous().view(B, S, -1)

        return self.w_out(out)